# 🇪🇹 Amharic Speech-to-Speech (S2S) Pipecat Pipeline
### Real-time, Streaming Voice Agent with VAD, ASR, LLM, and TTS

**Architecture:**
```
Audio Input (16kHz WAV / Mic)
       ↓
AmharicSTTService: snapwre/hohe-asr-amharic (CTC ASR)
       ↓ (Amharic Transcript)
AmharicLLMService: yosefw/gemma-2-2b-it-finetuned-amharic (Streaming 4-bit SFT LLM)
       ↓ (Streamed Tokens)
AmharicSentenceAggregator: Sentence boundary chunking (።, ?, !, \n)
       ↓ (Sentence Frames)
AmharicTTSService: gheero-Leyu/amharic-omnivoice-tts (Streaming PCM Synthesis)
       ↓
Audio Output (24 kHz PCM / Speaker)
```

> **Note:** All business logic, Pipecat service definitions, and audio handlers are packaged inside the modular `amharic_s2s` package. This notebook serves solely for loading and running the pipeline.

---  
## 1. Environment Setup & Dependency Installation

In [ ]:
# Install dependencies and package
!pip install -q pipecat-ai>=0.0.40 transformers accelerate bitsandbytes soundfile loguru silero-vad

# If running in Google Colab, clone or add repository root to sys.path
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("src"))

import torch
print("✓ PyTorch Version:", torch.__version__)
print("✓ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✓ GPU Device:", torch.cuda.get_device_name(0))
    print("✓ VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

---  
## 2. Load Models via Modular Model Loader

Loads the ASR (`snapwre/hohe-asr-amharic`), LLM (4-bit `yosefw/gemma-2-2b-it-finetuned-amharic`), and TTS (`gheero-Leyu/amharic-omnivoice-tts`).

In [ ]:
from amharic_s2s import load_all_models, default_config

# Customize config if desired
config = default_config
config.tts.diffusion_steps = 16  # 16 steps optimized for interactive streaming

# Load all models into memory
bundle = load_all_models(config)

---  
## 3. Interactive Streamed S2S Test

Provide an audio sample (or synthesized tone) to simulate real-time processing through the Pipecat streaming pipeline.

In [ ]:
import asyncio
import numpy as np
import IPython.display as ipd
from amharic_s2s.pipeline.runner import run_pipeline_with_audio_file

# Synthesize a sample test speech waveform or pass a path to a .wav file
sr = 16000
duration = 2.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
sample_audio = (0.3 * np.sin(2 * np.pi * 220.0 * t)).astype(np.float32)

# Run through the streaming pipeline
result = await run_pipeline_with_audio_file(bundle, sample_audio, orig_sr=sr, diffusion_steps=16)

print("=" * 60)
print(f"🎙️ ASR Transcript:   {result['transcript'] or '(No speech detected / synthetic tone)'}")
print(f"🤖 LLM Response:     {result['llm_response']}")
print(f"⚡ ASR Latency:      {result['asr_latency_sec']*1000:.1f} ms")
print(f"⚡ Time-To-1st-Token: {result['ttft_sec']*1000:.1f} ms")
print(f"⚡ Time-To-1st-Audio: {result['ttfa_sec']*1000:.1f} ms")
print(f"⏱️ Total Latency:    {result['total_latency_sec']*1000:.1f} ms")
print("=" * 60)

# Playback synthesized response
if result['audio_out'] is not None and len(result['audio_out']) > 0:
    ipd.display(ipd.Audio(result['audio_out'], rate=result['output_sr']))

---  
## 4. Run Pipecat Pipeline with Custom Audio Upload

In [ ]:
from google.colab import files

print("Upload an Amharic WAV audio file (or press Cancel):")
uploaded = files.upload()

for filename, audio_bytes in uploaded.items():
    print(f"Processing uploaded audio: {filename}...")
    res = await run_pipeline_with_audio_file(bundle, audio_bytes, diffusion_steps=16)
    print(f"Transcript:   {res['transcript']}")
    print(f"Response:     {res['llm_response']}")
    print(f"TTFA:         {res['ttfa_sec']*1000:.1f} ms")
    if res['audio_out'] is not None and len(res['audio_out']) > 0:
        ipd.display(ipd.Audio(res['audio_out'], rate=res['output_sr']))